# EDA to MLflow Pipeline

In [9]:
import pandas as pd
import sqlite3
import os

from rich.logging import RichHandler
import logging

In [10]:
# Configure logging with Rich
logging.basicConfig(
    level="INFO",
    format="%(message)s",
    datefmt="[%X]",
    handlers=[RichHandler(rich_tracebacks=True)]
)

logger = logging.getLogger("rich")

In [11]:
# 1) Load raw CSV Files
# 1) Load raw CSV Files
data_dir = os.path.join("data", "Corrected-Data")

patients_path = os.path.join(data_dir, "patients.csv")
visits_path = os.path.join(data_dir, "visits.csv")
billing_path = os.path.join(data_dir, "billing.csv")

for path in (patients_path, visits_path, billing_path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing data file: {path}")

patients = pd.read_csv(patients_path)
visits = pd.read_csv(visits_path)
billing = pd.read_csv(billing_path)

logger.info("patients shape: %s", patients.shape)
logger.info("visits shape: %s", visits.shape)
logger.info("billing shape: %s", billing.shape)




[11:16:25] INFO     patients shape: (5000, 7)                                                      ]8;id=6513132;file:///tmp/ipykernel_1864490/1631273376.py\1631273376.py]8;;\:]8;id=6513133;file:///tmp/ipykernel_1864490/1631273376.py#17\17]8;;\

           INFO     visits shape: (25000, 8)                                                       ]8;id=6513138;file:///tmp/ipykernel_1864490/1631273376.py\1631273376.py]8;;\:]8;id=6513139;file:///tmp/ipykernel_1864490/1631273376.py#18\18]8;;\

           INFO     billing shape: (25000, 7)                                                      ]8;id=6513144;file:///tmp/ipykernel_1864490/1631273376.py\1631273376.py]8;;\:]8;id=6513145;file:///tmp/ipykernel_1864490/1631273376.py#19\19]8;;\

In [12]:

# Create db folder, if it doesn't exits
os.makedirs("./db", exist_ok=True)

# Connect to sqlite and then create db
conn = sqlite3.connect("./db/hospital.db")
cursor = conn.cursor()
logger.info("Database connected: %s", conn)

           INFO     Database connected: <sqlite3.Connection object at 0x7d0c09668d60>               ]8;id=6513150;file:///tmp/ipykernel_1864490/2370726886.py\2370726886.py]8;;\:]8;id=6513151;file:///tmp/ipykernel_1864490/2370726886.py#7\7]8;;\

In [13]:

# write all three dataframes into sqlite tables
patients.to_sql("patients", conn, if_exists="replace", index=False)
visits.to_sql("visits", conn, if_exists="replace", index=False)
billing.to_sql("billing", conn, if_exists="replace", index=False)

logger.info("All three tables loaded into hospital db")

           INFO     All three tables loaded into hospital db                                        ]8;id=6513156;file:///tmp/ipykernel_1864490/3435725956.py\3435725956.py]8;;\:]8;id=6513157;file:///tmp/ipykernel_1864490/3435725956.py#6\6]8;;\

In [14]:
# 4) Quick preview of tables
logger.info("=== PATIENTS (first 3 rows) ===")
logger.info(pd.read_sql("select * from patients limit 3", conn).to_string())

logger.info("=== VISITS (first 3 rows) ===")
logger.info(pd.read_sql("select * from visits limit 3", conn).to_string())

logger.info("=== BILLING (first 3 rows) ===")
logger.info(pd.read_sql("select * from billing limit 3", conn).to_string())

           INFO     === PATIENTS (first 3 rows) ===                                                 ]8;id=6513162;file:///tmp/ipykernel_1864490/4173315521.py\4173315521.py]8;;\:]8;id=6513163;file:///tmp/ipykernel_1864490/4173315521.py#2\2]8;;\

           INFO        patient_id  age gender       city insurance_provider  chronic_flag           ]8;id=6513168;file:///tmp/ipykernel_1864490/4173315521.py\4173315521.py]8;;\:]8;id=6513169;file:///tmp/ipykernel_1864490/4173315521.py#3\3]8;;\
                    registration_date                                                                              
                    0           1   52      M  Bangalore            CareOne             1                          
                    5/6/2025                                                                                       
                    1           2   15      F     Mumbai            CareOne             0                          
                    12/27/2025                                                                                     
                    2           3   72      F  Bangalore         SecureLife             1                          
                    1/28/2025                                                                                      

           INFO     === VISITS (first 3 rows) ===                                                   ]8;id=6513174;file:///tmp/ipykernel_1864490/4173315521.py\4173315521.py]8;;\:]8;id=6513175;file:///tmp/ipykernel_1864490/4173315521.py#5\5]8;;\

           INFO        visit_id  patient_id  visit_date  department visit_type                      ]8;id=6513180;file:///tmp/ipykernel_1864490/4173315521.py\4173315521.py]8;;\:]8;id=6513181;file:///tmp/ipykernel_1864490/4173315521.py#6\6]8;;\
                    length_of_stay_hours risk_score  doctor_id                                                     
                    0         1           1  10/16/2025  Cardiology        OPD                                     
                    23.15        Low        186                                                                    
                    1         2           1   12/9/2025         ICU        ICU                                     
                    39.98       High        149                                                                    
                    2         3           1  10/29/2025          ER         ER                                     
                    20.97     Medium        122                                                                    

           INFO     === BILLING (first 3 rows) ===                                                  ]8;id=6513186;file:///tmp/ipykernel_1864490/4173315521.py\4173315521.py]8;;\:]8;id=6513187;file:///tmp/ipykernel_1864490/4173315521.py#8\8]8;;\

           INFO        bill_id  visit_id  billed_amount  approved_amount claim_status  payment_days ]8;id=6513192;file:///tmp/ipykernel_1864490/4173315521.py\4173315521.py]8;;\:]8;id=6513193;file:///tmp/ipykernel_1864490/4173315521.py#9\9]8;;\
                    billing_date                                                                                   
                    0        1         1       23550.00         21429.62         Paid           8.0                
                    2025-11-03                                                                                     
                    1        2         2       88539.01         19725.49      Pending           NaN                
                    2025-12-16                                                                                     
                    2        3         3       25949.30          1580.95     Rejected           NaN                
                    2025-11-20                                                                                     

## Operational Analytics

In [16]:
dept_workload = pd.read_sql("""
SELECT 
    department,
    COUNT(visit_id)                             AS total_visits,
    ROUND(AVG(length_of_stay_hours), 2)         AS avg_los_hours,
    ROUND(MAX(length_of_stay_hours), 2)         AS max_los_hours,
    SUM(CASE WHEN risk_score = 'High'
        THEN 1 ELSE 0 END)                      AS high_risk_visits
    FROM visits
    GROUP BY department
    ORDER BY total_visits DESC
""", conn)

logger.info(dept_workload.to_string(index=False))

# Which doctors handle the most high-risk patients?
doctor_risk = pd.read_sql("""
    SELECT 
        doctor_id,
        COUNT(visit_id)                              AS total_visits,
        SUM(CASE WHEN risk_score = 'High' 
            THEN 1 ELSE 0 END)                       AS high_risk_cases,
        ROUND(
            100.0 * SUM(CASE WHEN risk_score = 'High' 
            THEN 1 ELSE 0 END) / COUNT(visit_id), 1) AS high_risk_pct
    FROM visits
    GROUP BY doctor_id
    ORDER BY high_risk_cases DESC
    LIMIT 10
""", conn)

logger.info("Top 10 Doctors by High Risk Cases")
logger.info(doctor_risk.to_string(index=False))

# How many visits does each patient make on average?
visit_patterns = pd.read_sql("""
    SELECT 
        visit_frequency_bucket,
        COUNT(patient_id) AS num_patients
    FROM (
        SELECT 
            patient_id,
            COUNT(visit_id) AS visit_count,
            CASE 
                WHEN COUNT(visit_id) = 1     THEN '1 visit'
                WHEN COUNT(visit_id) BETWEEN 2 AND 3 THEN '2-3 visits'
                WHEN COUNT(visit_id) BETWEEN 4 AND 5 THEN '4-5 visits'
                ELSE '6+ visits'
            END AS visit_frequency_bucket
        FROM visits
        GROUP BY patient_id
    )
    GROUP BY visit_frequency_bucket
    ORDER BY num_patients DESC
""", conn)

logger.info("Patient Visit Frequency Distribution")
logger.info(visit_patterns.to_string(index=False))

# 8) Risk Score Distribution by Department
risk_by_dept = pd.read_sql("""
    SELECT 
        department,
        SUM(CASE WHEN risk_score = 'Low'    THEN 1 ELSE 0 END) AS low,
        SUM(CASE WHEN risk_score = 'Medium' THEN 1 ELSE 0 END) AS medium,
        SUM(CASE WHEN risk_score = 'High'   THEN 1 ELSE 0 END) AS high,
        COUNT(*) AS total
    FROM visits
    GROUP BY department
    ORDER BY high DESC
""", conn)

print("Risk Score Distribution by Department")
print(risk_by_dept.to_string(index=False))

[11:19:55] INFO      department  total_visits  avg_los_hours  max_los_hours  high_risk_visits      ]8;id=6513236;file:///tmp/ipykernel_1864490/1419670600.py\1419670600.py]8;;\:]8;id=6513237;file:///tmp/ipykernel_1864490/1419670600.py#14\14]8;;\
                        General          5757          14.01          36.54                24                      
                    Orthopedics          4474          23.02          52.99                54                      
                     Cardiology          4071          32.59          70.74               489                      
                             ER          3896          11.95          30.78               139                      
                            ICU          3415          56.86          78.42              2975                      
                      Neurology          3387          27.69          66.16               635                      

           INFO     Top 10 Doctors by High Risk Cases                                              ]8;id=6513243;file:///tmp/ipykernel_1864490/1419670600.py\1419670600.py]8;;\:]8;id=6513244;file:///tmp/ipykernel_1864490/1419670600.py#32\32]8;;\

           INFO      doctor_id  total_visits  high_risk_cases  high_risk_pct                       ]8;id=6513250;file:///tmp/ipykernel_1864490/1419670600.py\1419670600.py]8;;\:]8;id=6513251;file:///tmp/ipykernel_1864490/1419670600.py#33\33]8;;\
                           114           260               61           23.5                                       
                           130           264               60           22.7                                       
                           187           262               57           21.8                                       
                           126           244               55           22.5                                       
                           102           271               55           20.3                                       
                           189           240               54           22.5                                       
                           158           252               53           21.0                                       
                           171           260               52           20.0                                       
                           150           252               52           20.6                                       
                           194           267               51           19.1                                       

           INFO     Patient Visit Frequency Distribution                                           ]8;id=6513257;file:///tmp/ipykernel_1864490/1419670600.py\1419670600.py]8;;\:]8;id=6513258;file:///tmp/ipykernel_1864490/1419670600.py#57\57]8;;\

           INFO     visit_frequency_bucket  num_patients                                           ]8;id=6513264;file:///tmp/ipykernel_1864490/1419670600.py\1419670600.py]8;;\:]8;id=6513265;file:///tmp/ipykernel_1864490/1419670600.py#58\58]8;;\
                                 6+ visits          1816                                                           
                                4-5 visits          1776                                                           
                                2-3 visits          1267                                                           
                                   1 visit           141                                                           

Risk Score Distribution by Department
 department  low  medium  high  total
        ICU   10     430  2975   3415
  Neurology  875    1877   635   3387
 Cardiology 1063    2519   489   4071
         ER 1771    1986   139   3896
Orthopedics 3426     994    54   4474
    General 4590    1143    24   5757
